# BOTTLE приложение Parquet Tools

Литературный ноутбук со всем главным для приложения

## ENDPOINTS: Общая логика 

Какие url-ы что значат, как связаны и как это все работает

* main (main.tpl): главная форма, параметр fn - алиас обрабатываемого файла
    * этот параметр широко используется далее
    * без этого параметра используется в качестве начальной формы (где все кнопки кроме выбора файла заблокированы)
    * на форме - кнопки возможных действий с текущим файлом плюс кнопка выбора файла
* set_file (set_file.tpl): форма выбора файла 
* show_stats (show_stats.tpl): показать общую статистику файла fn (обязательный параметр)
* chart (show_byte_chart.tpl): отрисовать группы строк, блюмфильтры и проч файла fn (обязательный параметр)
* show_row_group_stats (get_row_group_stats.tpl): форм со статистикой по полю в пределах группы файла fn (обязательный параметр)
* col_chart (show_col_chart.tpl): отрисовка графика со статистикой колонки
* probe_bloom (show_bloom.tpl): отрисовка результата проверки блюм фильтра
    * работает только для локальных файлов (ибо через функцию parquet_bloom_probe()) 

## Использование модуля в Jupyter Notebook

Что хочется использовать в ноутбуке

* общая статистика по файлу
* статистика по колонке
* график файла
* график статистики по колонке
* пробу блюм фильтра

для этого функции модуля разделил на `prepXX()` и `showXX()`

Функции `prepXX()` содержат логику и могут использоваться в ноутбуках.

## Отладка и использование

Генерация модуля, выполнение тестов

In [ ]:
import asm, lit

In [ ]:
nbName = __session__.split("/")[-1].split(".ipynb")[0]
modName = nbName.replace("lit_","")
lit.genAsmNotebook(f"{nbName}.ipynb")  # порождает ноутбук xxx_asm.ipynb
asm.processFile(f"{nbName}_asm.ipynb") # порождает модуль xxx.py
!cp {nbName}.py {modName}.py # убираем префикс lit_
#asm.processFile(f"{nbName}_asm.ipynb","test") # порождает тест сьют test_xxx.py
#!pytest test_{nbName}.py # выполняем тесты

## Управление переводом

Это приложение двуязычное, поэтому нужна автоматизация/механизация переводов текстовых строк.

Ниже код, который позволяет собрать строки из питон файлов и шаблонов в YAML-файл для того, чтобы впоследствии их перевести, при этом сохранить уже переведённые строки.

Этот код явно универсальный и явно не является частью этого приложения, но пока не решил, куда его перенести.

### Сборка строк для перевода

In [ ]:
from pathlib import Path
import yaml

directories = [".", "views"]
extensions = {".py", ".tpl"}

files = [
    str(f)
    for d in directories
    for f in Path(d).iterdir()
    if f.is_file() and f.suffix in extensions
]

# gather strings to translate
trSet = set()
for fn in files:
    with open(fn,"r") as fp:
        for ln in fp:
            if ln.find('_("')>=0:
                trSet.add(ln.split('_("')[1].split('",')[0])

# gather translations
with open("rus_strings.yaml","r") as fp:
    RUS_STR = yaml.load(fp, Loader=yaml.FullLoader)

Path("rus_strings.yaml").replace("rus_strings.yaml.bak")

for ss in sorted(trSet):
    if ss not in RUS_STR:
        RUS_STR[ss] = ""

with open("rus_strings.yaml","w") as fp:
    fp.write(yaml.dump(RUS_STR, allow_unicode=True))
    

## The code

In [ ]:
from bottle import Bottle, template, static_file, route, post, request
import duckdb
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import struct
import os
from tpl_funcs import _

In [ ]:
from typing import List,Dict # для более корректных определений функций и их параметров

Создадим приложение

In [ ]:
app = Bottle()

In [ ]:
@app.route('/static/<filepath:path>')

## server_static:: для отдачи статических файлов, пока тестирую - использую относительный путь

### Параметры

In [ ]:
filepath: str:: имя файла (статического)

### Возвращаемое значение

In [ ]:
static_file:: собственно файл

### Тело

In [ ]:
#return static_file(filepath, root='/mnt/c/Users/mikhail.korolev/Documents/31.Parquet/bottle/static')
return static_file(filepath, root='static')

## mkd:: формирует строку с пробелами в качестве разделителей тысяч

### Параметры

In [ ]:
v: int:: значение

### Возвращаемое значение

In [ ]:
str:: строка с пробелами в качестве разделителей тысяч

### Тело

In [ ]:
return f"{v:,}".replace(',', ' ')

## get_parquet_metadata:: формирует датафрейм с метаданными паркет файла (для графика)

Старался делать универсально (для S3 и для локальных файлов), и поэтому получилось неаккуратно вычислить размер футера. Футер, как будто идёт сразу за последним увиденным смещением до конца файла.

### Параметры

In [ ]:
parquet_filename: str:: имя паркет файла

In [ ]:
conn: duckdb.DuckDBPyConnection:: база данных

### Возвращаемое значение

In [ ]:
pandas.DataFrame:: датафрейм, "заточенный" под последующее отображение в plotly

Колонки

- el_type: тип величины (одно из "column data", "column index", "bloom filter", "footer")
- column:  column name (None for footer)
- encodings: encodings for column
- rg_no:   row group number (None for footer)
- offset:  byte offset
- size:    size on disk in bytes

данные упорядочены по offset

### Тело

In [ ]:
# Row-group / column-chunk metadata from DuckDB
df = conn.execute(f"""
    SELECT
        row_group_id            AS rg_no,
        path_in_schema          AS column,
        encodings               AS encodings,
        data_page_offset        AS data_page_offset,
        dictionary_page_offset  AS dictionary_page_offset,
        index_page_offset       AS index_page_offset,
        bloom_filter_offset     AS bloom_filter_offset,
        bloom_filter_length     AS bloom_filter_length,
        total_compressed_size   AS total_compressed_size
    FROM pqmeta
""").fetchdf()

rows = []

maxOffset = 0 # максимальное смещение, которое встретилось

for r in df.itertuples(index=False):
    # ---- column data ------------------------------------------------
    # The column chunk starts at dictionary_page_offset if present,
    # otherwise at data_page_offset.
    if r.dictionary_page_offset is not None and not pd.isna(r.dictionary_page_offset):
        data_off = int(r.dictionary_page_offset)
    else:
        data_off = int(r.data_page_offset)

    data_size = int(r.total_compressed_size)
    rows.append({
        "el_type": "column data",
        "column":  r.column,
        "encodings": r.encodings,
        "rg_no":   int(r.rg_no),
        "offset":  data_off,
        "size":    data_size,
    })
    if data_off + data_size > maxOffset:
        maxOffset = data_off + data_size

    # ---- column index ----------------------------------------------
    if r.index_page_offset is not None and not pd.isna(r.index_page_offset):
        # offset index size isn't directly exposed by parquet_metadata;
        # the column index offset is index_page_offset and its length
        # can be found via the offset index. DuckDB doesn't expose the
        # length here, so we set size to 0 unless known.
        rows.append({
            "el_type": "column index",
            "column":  r.column,
            "encodings": None,
            "rg_no":   int(r.rg_no),
            "offset":  int(r.index_page_offset),
            "size":    0,   # length not exposed by parquet_metadata()
        })
        if int(r.index_page_offset) > maxOffset:
            maxOffset = int(r.index_page_offset)

    # ---- bloom filter ----------------------------------------------
    if r.bloom_filter_offset is not None and not pd.isna(r.bloom_filter_offset):
        bf_len = r.bloom_filter_length
        bf_len = int(bf_len) if bf_len is not None and not pd.isna(bf_len) else 0
        rows.append({
            "el_type": "bloom filter",
            "column":  r.column,
            "encodings": None,
            "rg_no":   int(r.rg_no),
            "offset":  int(r.bloom_filter_offset),
            "size":    bf_len,
        })
        if int(r.bloom_filter_offset) + bf_len > maxOffset:
            maxOffset = int(r.bloom_filter_offset) + bf_len

# ---- footer ---------------------------------------------------------
# Parquet file layout: ... <footer_metadata> <4-byte footer length> PAR1
file_size = conn.sql("select size from filename").df().values.tolist()[0][0]
footer_len = file_size - maxOffset
footer_offset = maxOffset

rows.append({
    "el_type": "footer",
    "column":  None,
    "encodings": None,
    "rg_no":   0, # let it be always within first row group
    "offset":  footer_offset,
    "size":    footer_len,
})

out = pd.DataFrame(rows, columns=["el_type", "column", "encodings", "rg_no", "offset", "size"])
out["row"] = out["rg_no"].mod(2).astype(int)
out = out.sort_values("offset").reset_index(drop=True)
return out

In [ ]:
@app.route('/static/<filepath:path>')

## server_static:: для отдачи статических файлов

### Параметры

In [ ]:
filepath: str:: путь к статическому файлу (пока тестирую - использую относительный путь)

### Возвращаемое значение

In [ ]:
static_file:: объект статический файл

### Тело

In [ ]:
#return static_file(filepath, root='/mnt/c/Users/mikhail.korolev/Documents/31.Parquet/bottle/static')
return static_file(filepath, root='static')

## getColumns:: возвращает список колонок паркет файла

### Параметры

In [ ]:
dbName: str:: путь к созданной базе мета данных 

### Возвращаемое значение

In [ ]:
List:: список имен колонок

### Тело

In [ ]:
conn = duckdb.connect(dbName)
cols = [ el[0] for el in conn.sql(f"SELECT name FROM pqschema order by 1").to_df().values.tolist() ]
conn.close()

return cols

In [ ]:
@app.get('/')

## entryPoint:: точка входа - отрисовка главной формы

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "main")

### Дополнительное описание

Изначально все кнопки кроме выбора файла не активны

### Тело

In [ ]:
lang = request.query.lang or "en"

ctx = {
    "fn": "",
    "lang": lang,
    "filename": _("no file selected yet",lang),
    "request": request,
}
return template('main',**ctx)

## prepMeta:: обработка файла - сохранение метаданных

### Параметры

In [ ]:
PQ_FILE: str:: путь к исходному паркет файлу 

In [ ]:
dbName: str:: путь к базе мета данных 

In [ ]:
S3_KEY: str:: S3 ключ 

In [ ]:
S3_SECRET: str:: S3 секрет 

### Дополнительное описание

Здесь происходит обработка метаданных файла

### Тело

In [ ]:
# удалим базу если существует
if os.path.exists(dbName):
    os.remove(dbName)
conn = duckdb.connect(dbName)

if PQ_FILE.lower().find("s3://")>=0: # работаем через HTTPS
    conn.sql("drop secret if exists ya_s3_secret")
    conn.sql(f"""
        CREATE SECRET ya_s3_secret (
            TYPE s3, PROVIDER config,
            KEY_ID '{S3_KEY}',
            SECRET '{S3_SECRET}',
            ENDPOINT "storage.yandexcloud.net", REGION 'ru-central1')
    """)

# save parquet filename and metadata - будем работать с первым файлом (в случае, если их несколько)
conn.sql(f"create table filename as SELECT filename as name, size, '(none)' as author, 1 as version FROM read_blob('{PQ_FILE}') limit 1")
PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0] # уточним название файла с которым работаем для последующей формы
conn.sql(f"create table pqmeta as SELECT * FROM parquet_metadata('{PQ_FILE}')")
conn.sql(f"create table pqschema as SELECT * FROM parquet_schema('{PQ_FILE}') WHERE num_children IS NULL")
author,version = conn.sql(f"SELECT created_by, format_version FROM parquet_file_metadata('{PQ_FILE}')").df().values[0]
conn.sql(f"update filename set author = '{author}', version = {version}")
conn.close()

In [ ]:
@app.post('/main')

In [ ]:
@app.get('/main')

## showMainForm:: показ главной формы с подготовленными данными

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "main")

### Дополнительное описание

Здесь происходит обработка метаданных файла и прописывание параметра fn для всех остальных форм

### Тело

In [ ]:
if request.query.fn=="": # работа с новым файлом
    PQ_FILE = request.forms.input_file # приезжает из формы
    PQ_ALIAS = request.forms.input_alias # приезжает из формы, передается в дальнейшие формы
    S3_KEY = request.forms.input_key
    S3_SECRET = request.forms.input_secret
    dbName = f"/tmp/{PQ_ALIAS}.duckdb"

    prepMeta( PQ_FILE, dbName, S3_KEY, S3_SECRET)
    
else: # продолжаем работать с тем же файлом - мета уже создана
    
    PQ_ALIAS = request.query.fn # приезжает из формы, передается в дальнейшие формы
    
    dbName = f"/tmp/{PQ_ALIAS}.duckdb"
    conn = duckdb.connect(dbName)
    PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]
    conn.close()

lang = request.query.lang # приезжает из формы
print("LANG",lang)

ctx = {
    "fn": PQ_ALIAS,
    "lang": lang,
    "filename": PQ_FILE,
    "request": request,
}
return template('main',**ctx)

## prepChart:: готовит график с байтовыми размерами частей файла

### Параметры

In [ ]:
dbName: str:: путь к созданной базе мета данных 

### Возвращаемое значение

In [ ]:
go.Figure:: созданный график

### Тело

In [ ]:
conn = duckdb.connect(dbName)

PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

mdf = get_parquet_metadata(PQ_FILE,conn)

conn.close()

fig = px.bar(
    mdf,
    x="size",
    y="row",
    base="offset",
    orientation="h",
    color="el_type",                       # <-- color by type, not element
    custom_data=["column", "el_type", "offset", "size", "encodings", "rg_no"],
)

fig.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>" # column name
        "type: %{customdata[1]}<br>"
        "offset: %{customdata[2]:,}<br>"
        "size: %{customdata[3]:,} bytes<br>"
        "encoding: %{customdata[4]}"
        "<extra>RG: %{customdata[5]}</extra>"
    )
)

fig.update_layout(
    barmode="overlay",
    showlegend=True,
    xaxis_title="Byte offset",
    yaxis_visible=False,
    bargap=0,
)

color_map = {
    "column data":  "#4C78A8",
    "column index": "#F58518",
    "bloom filter": "#54A24B",
    "footer":       "#B279A2",
}

for tr in fig.data:
    tr.marker.color = color_map.get(tr.name, tr.marker.color)

return fig

In [ ]:
@app.post('/chart')

## showChart:: показывает график с байтовыми размерами частей файла

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "show_byte_chart")

### Тело

In [ ]:
dbName = f"/tmp/{request.query.fn}.duckdb"

fig = prepChart(dbName)

conn = duckdb.connect(dbName)
PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]
conn.close()
    
plotly_html = fig.to_html(full_html=False, include_plotlyjs='cdn')

lang = request.query.lang # приезжает из формы

ctx = {
    "fn": request.query.fn,
    "lang": lang,
    "request": request,
    "filename": PQ_FILE,
    "chart_html": plotly_html
}

return template('show_byte_chart', **ctx )

## prepColChart:: формирует график статистикой колонки

### Параметры

In [ ]:
dbName: str:: путь к созданной базе мета данных 

In [ ]:
colName: str:: имя колонки 

In [ ]:
lang: str:: код языка 

### Возвращаемое значение

In [ ]:
go.Figure:: созданный график

### Тело

In [ ]:
conn = duckdb.connect(dbName)

PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

df = conn.sql(f"""
    select
        num_values as num_rows,
        stats_min_value as min, 
        stats_max_value as max, 
        stats_max_value::int - stats_min_value::int as span, 
        stats_null_count as nulls,
        row_group_id as rg
    from pqmeta 
    where 
        path_in_schema='{ colName }'
""").df()

conn.close()

df["rg_label"] = "RG " + df["rg"].astype(str)

fig = px.bar(
    df,
    x="rg_label",
    y="span",
    base="min",
    orientation="v",
    custom_data=["rg", "num_rows", "min", "max", "nulls"],
)

fig.update_traces(
    marker_color="#4C78A8",
    marker_line_color="#2A4A6B",
    marker_line_width=1,
    hovertemplate=(
        "<b>Row group %{customdata[0]}</b><br>"
        "rows:  %{customdata[1]:,}<br>"
        "min:   %{customdata[2]:,}<br>"
        "max:   %{customdata[3]:,}<br>"
        "nulls: %{customdata[4]:,}"
        "<extra></extra>"
    ),
)

fig.update_layout(
    title= colName + ": " +_("column stats (min/max) per row groups",lang),
    xaxis_title="",
    yaxis_title=_("value",lang),
    bargap=0.2,
    showlegend=False,
)

return fig

In [ ]:
@app.post('/col_chart')

## showColChart:: показывает график статистики колонки

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "show_col_chart")

### Тело

In [ ]:
dbName = f"/tmp/{request.query.fn}.duckdb"
lang = request.query.lang # приезжает из формы

conn = duckdb.connect(dbName)
PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

if request.forms.field_name: # вызов из формы

    colName = request.forms.field_name
    fig = prepColChart(dbName,colName,lang)
    
    cols = request.forms.colstr.split(",")
        
    plotly_html = fig.to_html(full_html=False, include_plotlyjs='cdn')

else:
    cols = [ el[0] for el in conn.sql(f"SELECT name FROM pqschema where type='INT32' order by 1").to_df().values.tolist() ]
    colName = cols[0]
    plotly_html = None

conn.close()

ctx = {
    "fn": request.query.fn,
    "lang": lang,
    "request": request,
    "filename": PQ_FILE,
    "columns": cols,
    "selected": colName,
    "chart_html": plotly_html
}

return template('show_col_chart', **ctx )

In [ ]:
@app.post('/set_file')

In [ ]:
@app.route('/set_file')

## setFile:: форма для установки имени файла - имя файла задается здесь

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "set_file")

### Тело

In [ ]:
lang = request.query.lang # приезжает из формы

return template('set_file', request=request, lang=lang)

## prepOverview:: подготовка общей статистики по файлу

### Параметры

In [ ]:
dbName: str:: путь к базе мета данных 

In [ ]:
lang: str:: код языка 

### Возвращаемое значение

In [ ]:
List:: список секций

### Тело

In [ ]:
conn = duckdb.connect(dbName)

PQ_FILE,size,author,version = conn.sql("select * from filename").df().values.tolist()[0]

grpNum = conn.sql("select max(row_group_id)+1 from pqmeta").df().values.tolist()[0][0]
rowNum = int(conn.sql("select sum(num_values) from pqmeta where column_id=0").df().values.tolist()[0][0])
rszList = conn.sql("select num_values, count(*) from pqmeta where column_id=0 group by 1").df().values.tolist()
compressions = str(conn.sql("SELECT list(DISTINCT compression) FROM pqmeta").fetchone()[0])
encList = conn.sql("""
SELECT
    encoding,
    string_agg(DISTINCT path_in_schema, ', ' ORDER BY path_in_schema) AS columns
FROM (
    SELECT
        path_in_schema,
        unnest(string_split(encodings, ',')) AS encoding
    FROM pqmeta
)
GROUP BY encoding
ORDER BY encoding;
""").df().values.tolist()
bloomColsList = conn.sql("""
SELECT DISTINCT path_in_schema AS column_name    FROM pqmeta
WHERE bloom_filter_length IS NOT NULL
ORDER BY column_name
""").df().values.tolist()

conn.close()

return [
    [ 
        _("General info",lang), # заголовок секции
        [ # содержимое секции - набор пар "ключ, значение"
            ("filename",PQ_FILE),
            ("rows",mkd(rowNum)),
            ("row groups",grpNum),
            ("file size",mkd(size)+" bytes"),
            ("compressions",compressions),
            ("created by",author),
            ("version",version)
        ]
    ],
    [
        _("Row Groups info",lang),
        [ (
            str(el[1])+" row groups have",
            mkd(el[0])+" rows"
          ) for el in rszList 
        ]
    ],
    [
        _("Encodings used",lang),
        [ (
            el[0], # encoding
            el[1] # columns
          ) for el in encList 
        ]
    ],
    [
        _("Bloom filter created",lang),
        [ (
            _("for columns",lang),
            ", ".join([ str(c[0]) for c in bloomColsList])
          ) 
        ]
    ]
    
]

In [ ]:
@app.post('/show_stats')

## showOverview:: форма для показа общей статистики по файлу

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "show_stats")

### Тело

In [ ]:
lang = request.query.lang # приезжает из формы

sections = prepOverview(f"/tmp/{request.query.fn}.duckdb",lang)

ctx = {
    "fn": request.query.fn,
    "lang": lang,
    "request": request,
    "sections": sections
}

return template('show_stats',**ctx)

## prepRowGroupStats:: подготовка данных для формы со статистикой по полю в Row Group

### Параметры

In [ ]:
dbName: str:: путь к базе мета данных 

In [ ]:
lang: str:: код языка 

In [ ]:
colName: str:: имя колонки 

In [ ]:
rowGroupInd: int:: номер группы строк 

### Возвращаемое значение

In [ ]:
Dict:: словарь со статистикой по полю группы строк

### Тело

In [ ]:
conn = duckdb.connect(dbName)

PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

resDf = conn.sql(f"""
    select
        '{_("value",lang)}' as {_("name",lang)},
        path_in_schema,
        column_id,
        type, 
        num_values,
        stats_min_value, 
        stats_max_value, 
        stats_null_count,
        stats_distinct_count,
        encodings, 
        compression,
        index_page_offset, 
        bloom_filter_offset,
        bloom_filter_length,
        data_page_offset,
        dictionary_page_offset,
        total_compressed_size,
        total_uncompressed_size
    from pqmeta 
    where 
        row_group_id={rowGroupInd} 
        and path_in_schema='{colName}'
""").df()
comments = [
    _("Meta column description",lang),        
    _("Column name",lang),
    _("column_id",lang),
    _("type",lang), 
    _("num_values",lang),
    _("stats_min_value",lang), 
    _("stats_max_value",lang), 
    _("stats_null_count",lang),
    _("stats_distinct_count",lang),
    _("encodings",lang), 
    _("compression",lang),
    _("index_page_offset",lang), 
    _("bloom_filter_offset",lang),
    _("bloom_filter_length",lang),
    _("data_page_offset",lang),
    _("dictionary_page_offset",lang),
    _("total_compressed_size",lang),
    _("total_uncompressed_size",lang),
]

return dict(zip(resDf.columns,zip(resDf.iloc[0],comments)))

In [ ]:
@app.post('/show_row_group_stats')

## showRowGroupStats:: показать статистику по полю в Row Group

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "get_row_group_stats")

### Тело

In [ ]:
dbName = f"/tmp/{request.query.fn}.duckdb"

conn = duckdb.connect(dbName)
PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

lang = request.query.lang # приезжает из формы

if request.forms.field_name: # вызов через форму, т.е. уже все заполнено
    
    rowGroupInd = int(request.forms.grp_numb)
    colName = request.forms.field_name

    resRow = prepRowGroupStats(dbName,lang,colName,rowGroupInd)
    
    cols = request.forms.colstr.split(",")
    grpNum = int(request.forms.grpnumbstr)
    
else: # первый вызов - из кнопки на главной форме, нужно заполнить поля формы
    
    grpNum = conn.sql("select max(row_group_id)+1 from pqmeta").df().values.tolist()[0][0]
    cols = [ el[0] for el in conn.sql(f"SELECT name FROM pqschema order by 1").to_df().values.tolist() ]
    resRow = None
    colName = cols[0]
    rowGroupInd = 0

conn.close()

ctx = {
    "title": "Row group stats",
    "fn": request.query.fn,
    "lang": lang,
    "request": request,
    "filename": PQ_FILE,
    "columns": cols,
    "grpnumb": grpNum,
    "selected": (colName,rowGroupInd),
    "statdict": resRow,
}

return template('get_row_group_stats',**ctx)

## probeBloomFilter:: проверка работы блюм фильтра

### Параметры

In [ ]:
dbName: str:: путь к базе мета данных 

In [ ]:
lang: str:: код языка 

In [ ]:
colName: str:: имя колонки 

In [ ]:
colValue: str:: значение колонки 

### Возвращаемое значение

In [ ]:
str:: строка со списком исключенных групп строк, инфой об исключении всех групп и т.п.

### Тело

In [ ]:
conn = duckdb.connect(dbName)

PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

grpNum = conn.sql("select max(row_group_id)+1 from pqmeta").df().values.tolist()[0][0]

res = conn.sql(f"select row_group_id from parquet_bloom_probe('{PQ_FILE}', '{colName}', {colValue}) where bloom_filter_excludes").df().values
conn.close()

if len(res)>0:
    grList = res.tolist()
    if len(grList)<grpNum:
        retStr = f"""{_("Following row groups will be excluded",lang)}: {",".join([str(r[0]) for r in grList])}"""
    else:
        retStr = _("All row groups will be excluded",lang)
else:
    retStr = _("No row groups will be excluded",lang)

return retStr

In [ ]:
@app.post('/probe_bloom')

## showProbeBloomFilter:: форма для проверки работы блюм фильтра

### Возвращаемое значение

In [ ]:
template:: данные для отрисовки (на основании шаблона "show_bloom")

### Тело

In [ ]:
dbName = f"/tmp/{request.query.fn}.duckdb"
lang = request.query.lang # приезжает из формы

conn = duckdb.connect(dbName)
PQ_FILE = conn.sql("select name from filename").df().values.tolist()[0][0]

if request.forms.input_probe: # вызов через форму, т.е. уже все заполнено

    colName = request.forms.field_name
    colValue = request.forms.input_probe
    cols = request.forms.colstr.split(",")

    retStr = probeBloomFilter(dbName,lang,colName,colValue)

else: # первый вызов
    
    colName = request.forms.field_name
    cols = [ el[0] for el in conn.sql(f"select distinct(path_in_schema) from pqmeta where bloom_filter_offset is not NULL order by 1").to_df().values.tolist() ]
    colValue = ""
    retStr = None

conn.close()

ctx = {
    "title": "Probe bloom",
    "fn": request.query.fn,
    "lang": lang,
    "request": request,
    "filename": PQ_FILE,
    "columns": cols,
    "probestr": colValue,
    "selected": colName,
    "resstr": retStr,
}

return template('show_bloom',**ctx)